<a href="https://colab.research.google.com/github/Sujitkar1511/Bank-additional/blob/main/AllModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# ==============================
# 1. INSTALL DEPENDENCIES
# ==============================
!pip install -q lightgbm catboost imbalanced-learn xgboost scikit-learn pandas numpy openpyxl

In [26]:
# ==============================
# 2. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix)

from imblearn.metrics import geometric_mean_score
from imblearn.over_sampling import ADASYN
from catboost import CatBoostClassifier

from sklearn.linear_model import LogisticRegression

In [16]:
# ==============================
# 3. LOAD DATASET
# ==============================
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')

In [17]:
# ==============================
# 4. PREPROCESSING
# ==============================
df = df.drop('duration', axis=1)

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

df['y'] = df['y'].map({'no': 0, 'yes': 1})

df = pd.get_dummies(df, drop_first=True)

In [18]:
# ==============================
# 5. SPLIT DATA
# ==============================
X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [19]:
# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [20]:
# ADASYN
adasyn = ADASYN(random_state=42)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_scaled, y_train)


In [21]:
# ==============================
# 6. METRIC FUNCTIONS
# ==============================
def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    print(f"Accuracy    : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision   : {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall      : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Specificity : {tn / (tn + fp):.4f}")
    print(f"F1 Score    : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"ROC-AUC     : {roc_auc_score(y_true, y_prob):.4f}")
    print(f"MCC         : {matthews_corrcoef(y_true, y_pred):.4f}")
    print(f"G-Mean      : {geometric_mean_score(y_true, y_pred):.4f}")
    print(f"Kappa       : {cohen_kappa_score(y_true, y_pred):.4f}")

# **CatBoostClassifier**

In [22]:
# ==============================
# 7. HYPERPARAMETER TUNING
# ==============================
params = {
    "iterations":[400,600],
    "depth":[4,5],
    "learning_rate":[0.03],
    "class_weights":[[1,2],[1,3]]
}

cv_small = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rscv = RandomizedSearchCV(
    CatBoostClassifier(verbose=0),
    params,
    n_iter=5,
    scoring='f1',
    cv=cv_small,
    n_jobs=-1
)

rscv.fit(X_train_ad, y_train_ad)
best = rscv.best_estimator_

print("🔥 Tuned Model Performance (Test Set)")
y_pred = best.predict(X_test_scaled)
y_prob = best.predict_proba(X_test_scaled)[:,1]
compute_metrics(y_test, y_pred, y_prob)

print("Best Params:", rscv.best_params_)


🔥 Tuned Model Performance (Test Set)
Accuracy    : 0.8891
Precision   : 0.5070
Recall      : 0.5485
Specificity : 0.9323
F1 Score    : 0.5269
ROC-AUC     : 0.8094
MCC         : 0.4646
G-Mean      : 0.7151
Kappa       : 0.4642
Best Params: {'learning_rate': 0.03, 'iterations': 400, 'depth': 5, 'class_weights': [1, 2]}


In [23]:
# ==============================
# 8. CROSS VALIDATION (MAIN)
# ==============================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [24]:
results1 = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best.predict(X_test_f)
    y_prob = best.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results1.append({
        "Fold": fold+1,
        "Classifier": "CatBoost",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [25]:
# ==============================
# 9. PRINT RESULTS
# ==============================
df_results = pd.DataFrame(results1)

print("\n===== Fold Results =====")
print(df_results.to_string(index=False))

print("\n===== Average Results =====")
print(df_results.mean(numeric_only=True))


# ==============================
# 10. SAVE TO EXCEL
# ==============================
df_results.to_excel("catboost_results.xlsx", index=False)

print("\n✅ Results saved to catboost_results.xlsx")


===== Fold Results =====
 Fold Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1   CatBoost  0.879582   0.468000 0.504310     0.927223 0.485477 0.683819 0.072777 0.796765 0.417770 0.417397           0.715767          10.296665
    2   CatBoost  0.875455   0.451292 0.489224     0.924487 0.469493 0.672519 0.075513 0.786692 0.399486 0.399069           0.706856           8.165116
    3   CatBoost  0.873513   0.446729 0.515086     0.919015 0.478478 0.688020 0.080985 0.799213 0.408256 0.406921           0.717051           8.169999
    4   CatBoost  0.876669   0.456863 0.502155     0.924213 0.478439 0.681248 0.075787 0.783601 0.409269 0.408682           0.713184           8.287469
    5   CatBoost  0.882010   0.478846 0.536638     0.925855 0.506098 0.704875 0.074145 0.809210 0.440266 0.439346           0.731246           7.619811
    6   CatBoost  0.888080   0.502982 0.545259     0.931601 0.

# Logistic Regression

In [27]:
# ==============================
# LOGISTIC REGRESSION TUNING
# ==============================
params_lr = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l1", "l2"]
}

rscv_lr = RandomizedSearchCV(
    LogisticRegression(max_iter=1000, solver='liblinear'),
    params_lr,
    n_iter=5,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_lr.fit(X_train_ad, y_train_ad)

best_lr = rscv_lr.best_estimator_

In [28]:
print("\n🔥 Logistic Regression (Tuned)")

y_pred_lr = best_lr.predict(X_test_scaled)
y_prob_lr = best_lr.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_lr, y_prob_lr)

print("Best Params (LR):", rscv_lr.best_params_)


🔥 Logistic Regression (Tuned)
Accuracy    : 0.7868
Precision   : 0.3056
Recall      : 0.7015
Specificity : 0.7977
F1 Score    : 0.4258
ROC-AUC     : 0.7977
MCC         : 0.3605
G-Mean      : 0.7480
Kappa       : 0.3189
Best Params (LR): {'penalty': 'l2', 'C': 1}


In [29]:
results_lr = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_lr.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_lr.predict(X_test_f)
    y_prob = best_lr.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_lr.append({
        "Fold": fold+1,
        "Classifier": "Logistic Regression",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [50]:
#df_cat = pd.DataFrame(results1)
df_lr  = pd.DataFrame(results_lr)

final_df = pd.concat([ df_lr])

In [51]:
print("\n===== ALL MODELS (Fold-wise) =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== ALL MODELS (Fold-wise) =====
 Fold          Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1 Logistic Regression  0.779315   0.291080 0.668103     0.793434 0.405494 0.728077 0.206566 0.788641 0.333273 0.294834           0.730769           1.638112
    2 Logistic Regression  0.780286   0.290598 0.659483     0.795622 0.403428 0.724361 0.204378 0.780269 0.329848 0.292837           0.727553           1.456582
    3 Logistic Regression  0.774703   0.284387 0.659483     0.789330 0.397403 0.721491 0.210670 0.782671 0.323006 0.284823           0.724406           1.411948
    4 Logistic Regression  0.773974   0.285189 0.668103     0.787415 0.399742 0.725310 0.212585 0.784804 0.326760 0.287192           0.727759           1.460981
    5 Logistic Regression  0.789026   0.305849 0.687500     0.801915 0.423358 0.742507 0.198085 0.800091 0.355832 0.316832           0.744708           1.34262

# ** DECISION TREE **

In [32]:
from sklearn.tree import DecisionTreeClassifier

In [33]:
# ==============================
# DECISION TREE TUNING
# ==============================
params_dt = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy"]
}

rscv_dt = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    params_dt,
    n_iter=10,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_dt.fit(X_train_ad, y_train_ad)

best_dt = rscv_dt.best_estimator_

In [34]:
print("\n🌳 Decision Tree (Tuned)")

y_pred_dt = best_dt.predict(X_test_scaled)
y_prob_dt = best_dt.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_dt, y_prob_dt)

print("Best Params (DT):", rscv_dt.best_params_)


🌳 Decision Tree (Tuned)
Accuracy    : 0.8515
Precision   : 0.3381
Recall      : 0.3319
Specificity : 0.9175
F1 Score    : 0.3350
ROC-AUC     : 0.6499
MCC         : 0.2514
G-Mean      : 0.5518
Kappa       : 0.2514
Best Params (DT): {'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'entropy'}


In [35]:
#Cross Validation (Decision Tree)

results_dt = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_dt.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_dt.predict(X_test_f)
    y_prob = best_dt.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_dt.append({
        "Fold": fold+1,
        "Classifier": "Decision Tree",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [48]:
#All Model
#df_cat = pd.DataFrame(results1)
#df_lr  = pd.DataFrame(results_lr)
df_dt  = pd.DataFrame(results_dt)

final_df = pd.concat([ df_dt])

In [49]:
#Print Comparison
print("\n===== ALL MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== ALL MODELS =====
 Fold    Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1 Decision Tree  0.862102   0.366667 0.308190     0.932421 0.334895 0.536062 0.067579 0.639190 0.259830 0.258615           0.620305           0.553874
    2 Decision Tree  0.853120   0.327628 0.288793     0.924761 0.306987 0.516783 0.075239 0.638309 0.225767 0.225207           0.606777           0.545098
    3 Decision Tree  0.845836   0.320755 0.329741     0.911354 0.325186 0.548189 0.088646 0.638369 0.238212 0.238183           0.620548           0.548773
    4 Decision Tree  0.859675   0.376623 0.375000     0.921204 0.375810 0.587751 0.078796 0.676327 0.296763 0.296762           0.648102           0.547155
    5 Decision Tree  0.857247   0.360360 0.344828     0.922298 0.352423 0.563945 0.077702 0.655610 0.272332 0.272248           0.633563           0.981173
    6 Decision Tree  0.849478   0.337500 0.349

# **RANDOM FOREST MODEL**

In [38]:
#Import Random Forest
from sklearn.ensemble import RandomForestClassifier

In [39]:
#Hyperparameter Tuning
# ==============================
# RANDOM FOREST TUNING
# ==============================
params_rf = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

rscv_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    params_rf,
    n_iter=10,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_rf.fit(X_train_ad, y_train_ad)

best_rf = rscv_rf.best_estimator_

In [40]:
#Test Performance
print("\n🌲 Random Forest (Tuned)")

y_pred_rf = best_rf.predict(X_test_scaled)
y_prob_rf = best_rf.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_rf, y_prob_rf)

print("Best Params (RF):", rscv_rf.best_params_)


🌲 Random Forest (Tuned)
Accuracy    : 0.8915
Precision   : 0.5246
Recall      : 0.3912
Specificity : 0.9550
F1 Score    : 0.4481
ROC-AUC     : 0.7913
MCC         : 0.3945
G-Mean      : 0.6112
Kappa       : 0.3894
Best Params (RF): {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}


In [41]:
#Cross Validation (Random Forest)
results_rf = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling (optional for RF, but keeping consistent)
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_rf.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_rf.predict(X_test_f)
    y_prob = best_rf.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_rf.append({
        "Fold": fold+1,
        "Classifier": "Random Forest",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [46]:
#Combine ALL Models
#df_cat = pd.DataFrame(results1)
#df_lr  = pd.DataFrame(results_lr)
#df_dt  = pd.DataFrame(results_dt)
df_rf  = pd.DataFrame(results_rf)

#final_df = pd.concat([df_cat, df_lr, df_dt, df_rf])

final_df = pd.concat([df_rf])

In [47]:
print("\n===== ALL MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== ALL MODELS =====
 Fold    Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1 Random Forest  0.890265   0.516760 0.398707     0.952668 0.450122 0.616308 0.047332 0.780992 0.394348 0.390296           0.675687          10.185901
    2 Random Forest  0.880068   0.456395 0.338362     0.948837 0.388614 0.566613 0.051163 0.760484 0.328207 0.323749           0.643600           8.313356
    3 Random Forest  0.877640   0.444751 0.346983     0.945007 0.389831 0.572626 0.054993 0.784531 0.326058 0.322983           0.645995           7.878246
    4 Random Forest  0.886380   0.494565 0.392241     0.949111 0.437500 0.610148 0.050889 0.766892 0.378362 0.375242           0.670676           6.223018
    5 Random Forest  0.895606   0.549708 0.405172     0.957866 0.466501 0.622977 0.042134 0.782665 0.415976 0.410109           0.681519           6.544795
    6 Random Forest  0.886866   0.497340 0.403

# **Gradient Boosting (GBM)**

In [52]:
#Import GBM

from sklearn.ensemble import GradientBoostingClassifier

In [53]:
#Hyperparameter Tuning

# ==============================
# GRADIENT BOOSTING TUNING
# ==============================
params_gbm = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rscv_gbm = RandomizedSearchCV(
    GradientBoostingClassifier(),
    params_gbm,
    n_iter=10,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_gbm.fit(X_train_ad, y_train_ad)

best_gbm = rscv_gbm.best_estimator_

In [54]:
#Test Performance

print("\n Gradient Boosting (Tuned)")

y_pred_gbm = best_gbm.predict(X_test_scaled)
y_prob_gbm = best_gbm.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_gbm, y_prob_gbm)

print("Best Params (GBM):", rscv_gbm.best_params_)


🌿 Gradient Boosting (Tuned)
Accuracy    : 0.9003
Precision   : 0.6047
Recall      : 0.3330
Specificity : 0.9724
F1 Score    : 0.4295
ROC-AUC     : 0.8036
MCC         : 0.4002
G-Mean      : 0.5690
Kappa       : 0.3798
Best Params (GBM): {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 5, 'learning_rate': 0.1}


In [55]:
#Cross Validation (GBM)

results_gbm = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_gbm.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_gbm.predict(X_test_f)
    y_prob = best_gbm.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_gbm.append({
        "Fold": fold+1,
        "Classifier": "Gradient Boosting",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [58]:
#Combine ALL Models
#df_cat = pd.DataFrame(results1)
#df_lr  = pd.DataFrame(results_lr)
#df_dt  = pd.DataFrame(results_dt)
#df_rf  = pd.DataFrame(results_rf)
df_gbm = pd.DataFrame(results_gbm)

final_df = pd.concat([df_gbm])

In [59]:
#Print Comparison

print("\n===== ALL MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== ALL MODELS =====
 Fold        Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1 Gradient Boosting  0.898762   0.588679 0.336207     0.970178 0.427984 0.571122 0.029822 0.791798 0.394813 0.376957           0.653192          57.298545
    2 Gradient Boosting  0.891964   0.540773 0.271552     0.970725 0.361549 0.513422 0.029275 0.780997 0.331578 0.309548           0.621138          55.704762
    3 Gradient Boosting  0.889294   0.515267 0.290948     0.965253 0.371901 0.529942 0.034747 0.795565 0.331901 0.316313           0.628101          57.178921
    4 Gradient Boosting  0.899490   0.592593 0.344828     0.969904 0.435967 0.578316 0.030096 0.782873 0.402057 0.384999           0.657366          57.072791
    5 Gradient Boosting  0.899247   0.589091 0.349138     0.969083 0.438430 0.581673 0.030917 0.810554 0.403064 0.387041           0.659111          56.052825
    6 Gradient Boostin

# **XGBoost**

In [60]:
from xgboost import XGBClassifier

In [61]:
#Hyperparameter Tuning

# ==============================
# XGBOOST TUNING
# ==============================
params_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

rscv_xgb = RandomizedSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    params_xgb,
    n_iter=10,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_xgb.fit(X_train_ad, y_train_ad)

best_xgb = rscv_xgb.best_estimator_

In [62]:
print("\n⚡ XGBoost (Tuned)")

y_pred_xgb = best_xgb.predict(X_test_scaled)
y_prob_xgb = best_xgb.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_xgb, y_prob_xgb)

print("Best Params (XGB):", rscv_xgb.best_params_)


⚡ XGBoost (Tuned)
Accuracy    : 0.9023
Precision   : 0.6284
Recall      : 0.3244
Specificity : 0.9756
F1 Score    : 0.4279
ROC-AUC     : 0.8007
MCC         : 0.4053
G-Mean      : 0.5625
Kappa       : 0.3803
Best Params (XGB): {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.8}


In [63]:
results_xgb = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_xgb.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_xgb.predict(X_test_f)
    y_prob = best_xgb.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_xgb.append({
        "Fold": fold+1,
        "Classifier": "XGBoost",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [66]:
#df_cat = pd.DataFrame(results1)
#df_lr  = pd.DataFrame(results_lr)
#df_dt  = pd.DataFrame(results_dt)
#df_rf  = pd.DataFrame(results_rf)
#df_gbm = pd.DataFrame(results_gbm)
df_xgb = pd.DataFrame(results_xgb)

final_df = pd.concat([ df_xgb])

In [67]:
print("\n===== ALL MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== ALL MODELS =====
 Fold Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1    XGBoost  0.897062   0.582645 0.303879     0.972367 0.399433 0.543583 0.027633 0.786211 0.371401 0.349172           0.638123           7.292342
    2    XGBoost  0.890265   0.528037 0.243534     0.972367 0.333333 0.486626 0.027633 0.776655 0.307567 0.282296           0.607951           6.178258
    3    XGBoost  0.891721   0.538462 0.271552     0.970451 0.361032 0.513350 0.029549 0.786472 0.330537 0.308828           0.621002           7.037250
    4    XGBoost  0.899976   0.604839 0.323276     0.973187 0.421348 0.560899 0.026813 0.771697 0.394036 0.372072           0.648232          12.683976
    5    XGBoost  0.902161   0.615094 0.351293     0.972093 0.447188 0.584371 0.027907 0.804914 0.416721 0.397875           0.661693           8.204955
    6    XGBoost  0.902403   0.622047 0.340517     0.973735 0.44

# **LightGBM**

In [68]:
from lightgbm import LGBMClassifier

In [69]:
# ==============================
# LIGHTGBM TUNING
# ==============================
params_lgb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [31, 50],
    "max_depth": [-1, 10],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

rscv_lgb = RandomizedSearchCV(
    LGBMClassifier(class_weight='balanced'),
    params_lgb,
    n_iter=10,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_lgb.fit(X_train_ad, y_train_ad)

best_lgb = rscv_lgb.best_estimator_

[LightGBM] [Info] Number of positive: 29184, number of negative: 29238
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014765 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8217
[LightGBM] [Info] Number of data points in the train set: 58422, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [70]:
print("\n LightGBM (Tuned)")

y_pred_lgb = best_lgb.predict(X_test_scaled)
y_prob_lgb = best_lgb.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_lgb, y_prob_lgb)

print("Best Params (LGB):", rscv_lgb.best_params_)


 LightGBM (Tuned)
Accuracy    : 0.9009
Precision   : 0.6167
Recall      : 0.3190
Specificity : 0.9748
F1 Score    : 0.4205
ROC-AUC     : 0.8018
MCC         : 0.3965
G-Mean      : 0.5576
Kappa       : 0.3722
Best Params (LGB): {'subsample': 0.8, 'num_leaves': 50, 'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [76]:
results_lgb = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling (optional but keeping consistent)
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_lgb.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_lgb.predict(X_test_f)
    y_prob = best_lgb.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_lgb.append({
        "Fold": fold+1,
        "Classifier": "LightGBM",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

[LightGBM] [Info] Number of positive: 32580, number of negative: 32893
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.034460 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8233
[LightGBM] [Info] Number of data points in the train set: 65473, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Info] Number of positive: 32368, number of negative: 32893
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.036939 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8214
[LightGBM] [Info] Number of data points in the train set: 65261, number of used features: 45
[LightGBM] [Info] 

In [77]:
#df_cat = pd.DataFrame(results1)
#df_lr  = pd.DataFrame(results_lr)
#df_dt  = pd.DataFrame(results_dt)
#df_rf  = pd.DataFrame(results_rf)
#df_gbm = pd.DataFrame(results_gbm)
#df_xgb = pd.DataFrame(results_xgb)
df_lgb = pd.DataFrame(results_lgb)

final_df = pd.concat([ df_lgb])

In [78]:
print("\n===== ALL MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== ALL MODELS =====
 Fold Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1   LightGBM  0.899005   0.596774 0.318966     0.972640 0.415730 0.556991 0.027360 0.786994 0.387580 0.365976           0.645803           4.723939
    2   LightGBM  0.891721   0.539474 0.265086     0.971272 0.355491 0.507416 0.028728 0.773454 0.326795 0.303813           0.618179           3.545590
    3   LightGBM  0.890750   0.528926 0.275862     0.968810 0.362606 0.516970 0.031190 0.791597 0.328951 0.309263           0.622336           2.263850
    4   LightGBM  0.898276   0.594937 0.303879     0.973735 0.402282 0.543965 0.026265 0.770387 0.376914 0.353000           0.638807           1.955662
    5   LightGBM  0.899733   0.604938 0.316810     0.973735 0.415842 0.555418 0.026265 0.804608 0.389871 0.366810           0.645272           1.939265
    6   LightGBM  0.901918   0.614504 0.346983     0.972367 0.44

# **K Nearest Neighbour(KNN)**

In [80]:
from sklearn.neighbors import KNeighborsClassifier

In [81]:
# ==============================
# KNN TUNING
# ==============================
params_knn = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

rscv_knn = RandomizedSearchCV(
    KNeighborsClassifier(),
    params_knn,
    n_iter=8,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_knn.fit(X_train_ad, y_train_ad)

best_knn = rscv_knn.best_estimator_

In [82]:
print("\n KNN (Tuned)")

y_pred_knn = best_knn.predict(X_test_scaled)
y_prob_knn = best_knn.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_knn, y_prob_knn)

print("Best Params (KNN):", rscv_knn.best_params_)


 KNN (Tuned)
Accuracy    : 0.8274
Precision   : 0.3046
Recall      : 0.4149
Specificity : 0.8798
F1 Score    : 0.3513
ROC-AUC     : 0.6957
MCC         : 0.2585
G-Mean      : 0.6041
Kappa       : 0.2544
Best Params (KNN): {'weights': 'distance', 'n_neighbors': 3, 'metric': 'manhattan'}


In [83]:
results_knn = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling (VERY IMPORTANT for KNN ⚠️)
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_knn.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_knn.predict(X_test_f)
    y_prob = best_knn.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_knn.append({
        "Fold": fold+1,
        "Classifier": "KNN",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [84]:
#df_cat = pd.DataFrame(results1)
#df_lr  = pd.DataFrame(results_lr)
#df_dt  = pd.DataFrame(results_dt)
#df_rf  = pd.DataFrame(results_rf)
#df_gbm = pd.DataFrame(results_gbm)
#df_xgb = pd.DataFrame(results_xgb)
#df_lgb = pd.DataFrame(results_lgb)
df_knn = pd.DataFrame(results_knn)

final_df = pd.concat([df_knn])

In [85]:
print("\n===== ALL MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== ALL MODELS =====
 Fold Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1        KNN  0.829328   0.311811 0.426724     0.880438 0.360328 0.612947 0.119562 0.687434 0.268933 0.264595           0.653581           0.002802
    2        KNN  0.825929   0.293638 0.387931     0.881532 0.334262 0.584785 0.118468 0.679720 0.239368 0.236334           0.634732           0.003000
    3        KNN  0.824472   0.288744 0.381466     0.880711 0.328691 0.579621 0.119289 0.673525 0.232895 0.229944           0.631088           0.003754
    4        KNN  0.828842   0.306581 0.411638     0.881806 0.351426 0.602482 0.118194 0.682500 0.258939 0.255260           0.646722           0.002661
    5        KNN  0.833940   0.329193 0.456897     0.881806 0.382671 0.634739 0.118194 0.706445 0.294850 0.289652           0.669351           0.003428
    6        KNN  0.828842   0.310236 0.424569     0.880164 0.35

# **Multi Layer Perceptron (MLP)**

In [86]:
from sklearn.neural_network import MLPClassifier

In [87]:
# ==============================
# MLP TUNING
# ==============================
params_mlp = {
    "hidden_layer_sizes": [(50,), (100,), (50,50)],
    "activation": ["relu", "tanh"],
    "alpha": [0.0001, 0.001],
    "learning_rate": ["constant", "adaptive"]
}

rscv_mlp = RandomizedSearchCV(
    MLPClassifier(max_iter=500, random_state=42),
    params_mlp,
    n_iter=8,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_mlp.fit(X_train_ad, y_train_ad)

best_mlp = rscv_mlp.best_estimator_

In [88]:
print("\n MLP (Tuned)")

y_pred_mlp = best_mlp.predict(X_test_scaled)
y_prob_mlp = best_mlp.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_mlp, y_prob_mlp)

print("Best Params (MLP):", rscv_mlp.best_params_)


🧠 MLP (Tuned)
Accuracy    : 0.8263
Precision   : 0.2930
Recall      : 0.3836
Specificity : 0.8825
F1 Score    : 0.3322
ROC-AUC     : 0.6823
MCC         : 0.2373
G-Mean      : 0.5818
Kappa       : 0.2345
Best Params (MLP): {'learning_rate': 'adaptive', 'hidden_layer_sizes': (100,), 'alpha': 0.001, 'activation': 'relu'}


In [89]:
results_mlp = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling (VERY IMPORTANT for MLP ⚠️)
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_mlp.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_mlp.predict(X_test_f)
    y_prob = best_mlp.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_mlp.append({
        "Fold": fold+1,
        "Classifier": "MLP",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [90]:
#df_cat = pd.DataFrame(results1)
#df_lr  = pd.DataFrame(results_lr)
#df_dt  = pd.DataFrame(results_dt)
#df_rf  = pd.DataFrame(results_rf)
#df_gbm = pd.DataFrame(results_gbm)
#df_xgb = pd.DataFrame(results_xgb)
#df_lgb = pd.DataFrame(results_lgb)
#df_knn = pd.DataFrame(results_knn)
df_mlp = pd.DataFrame(results_mlp)

final_df = pd.concat([ df_mlp])

In [91]:
print("\n===== ALL MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== ALL MODELS =====
 Fold Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1        MLP  0.819859   0.282813 0.390086     0.874419 0.327899 0.584037 0.125581 0.682310 0.230844 0.226930           0.632252          92.291785
    2        MLP  0.830541   0.288809 0.344828     0.892202 0.314342 0.554667 0.107798 0.652816 0.219645 0.218527           0.618515          89.166057
    3        MLP  0.814761   0.277860 0.403017     0.867031 0.328936 0.591125 0.132969 0.687149 0.230930 0.225675           0.635024          69.946360
    4        MLP  0.835882   0.314035 0.385776     0.893023 0.346228 0.586947 0.106977 0.676649 0.255272 0.253517           0.639400          63.062081
    5        MLP  0.837825   0.321678 0.396552     0.893844 0.355212 0.595362 0.106156 0.706184 0.265500 0.263612           0.645198          89.584066
    6        MLP  0.824958   0.305008 0.433190     0.874692 0.35

# ** Bagging Classifier**

In [92]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

In [93]:
# ==============================
# BAGGING TUNING
# ==============================
params_bag = {
    "n_estimators": [50, 100],
    "max_samples": [0.8, 1.0],
    "max_features": [0.8, 1.0]
}

rscv_bag = RandomizedSearchCV(
    BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        random_state=42
    ),
    params_bag,
    n_iter=6,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_bag.fit(X_train_ad, y_train_ad)

best_bag = rscv_bag.best_estimator_

In [94]:
print("\n🎯 Bagging Classifier (Tuned)")

y_pred_bag = best_bag.predict(X_test_scaled)
y_prob_bag = best_bag.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_bag, y_prob_bag)

print("Best Params (Bagging):", rscv_bag.best_params_)


🎯 Bagging Classifier (Tuned)
Accuracy    : 0.8915
Precision   : 0.5286
Recall      : 0.3384
Specificity : 0.9617
F1 Score    : 0.4126
ROC-AUC     : 0.7849
MCC         : 0.3668
G-Mean      : 0.5704
Kappa       : 0.3560
Best Params (Bagging): {'n_estimators': 100, 'max_samples': 0.8, 'max_features': 0.8}


In [95]:
results_bag = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling (optional)
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_bag.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_bag.predict(X_test_f)
    y_prob = best_bag.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_bag.append({
        "Fold": fold+1,
        "Classifier": "Bagging",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [96]:

df_bag = pd.DataFrame(results_bag)

final_df = pd.concat([
     df_bag
])

In [97]:
print("\n===== ALL MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== ALL MODELS =====
 Fold Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1    Bagging  0.893178   0.539216 0.355603     0.961423 0.428571 0.584710 0.038577 0.769295 0.382211 0.372378           0.658513          23.557584
    2    Bagging  0.885409   0.486395 0.308190     0.958687 0.377309 0.543560 0.041313 0.763504 0.327735 0.317685           0.633438          23.066938
    3    Bagging  0.884924   0.483974 0.325431     0.955951 0.389175 0.557760 0.044049 0.775675 0.336225 0.328333           0.640691          23.913094
    4    Bagging  0.888808   0.509259 0.355603     0.956498 0.418782 0.583210 0.043502 0.762280 0.366539 0.359444           0.656051          23.522064
    5    Bagging  0.895849   0.555556 0.377155     0.961696 0.449294 0.602253 0.038304 0.783963 0.403122 0.394095           0.669426          23.723793
    6    Bagging  0.890265   0.518750 0.357759     0.957866 0.42

# **Print All Model**

In [98]:
df_cat = pd.DataFrame(results1)
df_lr  = pd.DataFrame(results_lr)
df_dt  = pd.DataFrame(results_dt)
df_rf  = pd.DataFrame(results_rf)
df_gbm = pd.DataFrame(results_gbm)
df_xgb = pd.DataFrame(results_xgb)
df_lgb = pd.DataFrame(results_lgb)
df_knn = pd.DataFrame(results_knn)
df_mlp = pd.DataFrame(results_mlp)
df_bag = pd.DataFrame(results_bag)

final_df = pd.concat([
    df_cat, df_lr, df_dt, df_rf, df_gbm,
    df_xgb, df_lgb, df_knn, df_mlp, df_bag
])


In [102]:
print("\n===== ALL MODELS =====")
print(final_df.round(4).to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== ALL MODELS =====
 Fold          Classifier  Accuracy  Precision  Recall  Specificity     F1     GM    FPR    AUC    MCC  Kappa  Balanced Accuracy  Training Time (s)
    1            CatBoost    0.8796     0.4680  0.5043       0.9272 0.4855 0.6838 0.0728 0.7968 0.4178 0.4174             0.7158            10.2967
    2            CatBoost    0.8755     0.4513  0.4892       0.9245 0.4695 0.6725 0.0755 0.7867 0.3995 0.3991             0.7069             8.1651
    3            CatBoost    0.8735     0.4467  0.5151       0.9190 0.4785 0.6880 0.0810 0.7992 0.4083 0.4069             0.7171             8.1700
    4            CatBoost    0.8767     0.4569  0.5022       0.9242 0.4784 0.6812 0.0758 0.7836 0.4093 0.4087             0.7132             8.2875
    5            CatBoost    0.8820     0.4788  0.5366       0.9259 0.5061 0.7049 0.0741 0.8092 0.4403 0.4393             0.7312             7.6198
    6            CatBoost    0.8881     0.5030  0.5453       0.9316 0.5233 0.7127 0.0684

In [103]:
final_df.to_excel("all_models_results.xlsx", index=False)
print("\n✅ Saved: all_models_results.xlsx")


✅ Saved: all_models_results.xlsx


In [105]:
final_df.round(4).to_excel("all_models_results_4decimal.xlsx", index=False)

In [106]:
files.download("all_models_results_4decimal.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>